# Setup 

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import bibtexparser
sys.path.append(os.getcwd())
from sonicsocialgenre import loader

# Load Data 

In [ ]:
olddat = loader.load_survey_data(
    "data/original_survey.csv",skiprows=[0,2,3], map=loader.datamaps["old"]
)
newdat = loader.load_survey_data(
    "data/new_survey.csv",skiprows=[0,2], map=loader.datamaps["new"]
)

## Test that keys in data and bib match up

In [ ]:
library = bibtexparser.parse_file('data/articles.bib')

bib_keys = library.entries_dict.keys()
csv_keys = newdat.index

# Any duplicate keys in either?
def duplicate_count(keys):
    from collections import Counter
    counts = Counter(keys)
    dups = [k for k, n in counts.items() if n > 1]
    return len(dups)
assert(duplicate_count(bib_keys) == 0)
assert(duplicate_count(csv_keys) == 0)

# Key sets match? 
assert(len(set(bib_keys).symmetric_difference(set(csv_keys))) == 0)


# Queries

In [ ]:
# Return the set of articles in the new data that use GTZAN on its own
newdat[(newdat.datasets.GTZAN > 0) & (newdat.datasets.sum(axis=1) < 2)]

In [ ]:
# Who engages with MIR crit and uses GTZAN?
newdat[(newdat.datasets.GTZAN > 0) & (newdat.datasets.sum(axis=1) < 2) &  (newdat.curiosity["Engages Critical MIR"] > 0)]

In [ ]:
# Return the set of articles in the data using only accuracy
newdat[(newdat.fom.Accuracy > 0) & (newdat.fom.sum(axis=1) < 2)]


In [ ]:
# Using only classify and accuracy 
newdat[(newdat.doe.Classify > 0) & (newdat.doe.sum(axis=1)< 2) & (newdat.fom.Accuracy > 0) & (newdat.fom.sum(axis=1) < 2)]


In [ ]:
pc_acc = newdat[(newdat.fom.Accuracy > 0) & (newdat.year < 2023)].shape[0] / newdat[(newdat.year < 2023)].shape[0]
pc_clas = newdat[(newdat.doe.Classify > 0) & (newdat.year < 2023)].shape[0] / newdat[(newdat.year < 2023)].shape[0]
print(pc_acc, pc_clas)

In [ ]:
# Histogram of number of datasets used; looks weird to me – should check it out
# newdat.aggregates['Number public'].hist()
#  or
newdat.datasets.sum(axis=1).hist(bins=range(8), align="left")
plt.xticks(range(7))

In [ ]:
# Histogram of public datasets for papers with 'non western'

newdat[newdat.data_types["Non-Western?"] > 0].datasets.sum(axis=1).hist(
    align="left", bins=range(5)
)
plt.xticks(range(4))

## Publications by Year

In [ ]:
# Publication years – make a new column 

path = '/Users/owen/Documents/SSG Prism pres/'

import re

def add_year_col(df):
    expr = r".+([0-9]{4})[a-z]?"
    df['year'] = [int(re.match(expr,e).groups()[0]) for e in list(df.index)]
    return df

newdat = add_year_col(newdat)
olddat = add_year_col(olddat)

all_years_df = pd.concat([newdat[newdat.year < 2023].year, olddat.year])
cm = 1/2.54
figsize = (25*cm, 15*cm)
f, ax = plt.subplots(1,1, figsize=figsize)
all_years_df.hist(bins=range(1995,2025), align='mid', width=0.5, zorder=3,ax=ax)
print(ax.containers)

for bar in ax.containers[0][:18]:
    print(bar)
    plt.setp(bar, color='lightsteelblue')

plt.grid(which='both',linestyle=':', zorder=0)
plt.xticks(range(1995,2023),rotation=45)
plt.savefig(path + 'pubyears.png')


## Dataset Usage

In [ ]:
## TODO: fix labels 
f, ax = plt.subplots(1,1,figsize=figsize)
newdat.datasets.apply(lambda x: x.sum()).sort_values(ascending=False)[0:10].plot(kind='bar', ax = ax)
ax.tick_params(axis='x', labelrotation=45,labelsize=14)
f.align_labels()
plt.tight_layout()
plt.savefig(path + 'dataset_usage.png')


## FoM Usage

In [ ]:
f, ax = plt.subplots(nrows=1,ncols=2, figsize=figsize)
olddat.fom.apply(lambda x: x.sum()).plot(kind='bar', title='Figures of Merit up to 2013', ax=ax[0])
newdat.fom.apply(lambda x: x.sum()).plot(kind='bar', title='Figures of Merit since 2013', ax=ax[1])
for a in ax: 
    a.tick_params(axis='x', labelrotation=90)
f.align_labels()
plt.tight_layout()
plt.savefig(path + 'fom.png')

# Publication Types

In [ ]:

newdat['pubtype'] = [library.entries_dict[i].entry_type for i in newdat.index]

pub_piv = newdat[(newdat.year >= 2012) & (newdat.year <= 2022)].groupby('year')['pubtype'].value_counts().unstack()

fig,ax = plt.subplots(figsize=(10,7))
pub_piv.plot.bar(stacked=True,ylabel='Publications',ax = ax,zorder=3)



In [ ]:
# Let's see if we can make a reasonable list of unique publication venues

def venues(library):
    lookup = {
        'article':['journaltitle'], 
        'inproceedings': ['booktitle','eventtitle']
    }
    for e in library.entries: 
        # if e.entry_type in lookup.keys:
        ven_key = lookup.get(e.entry_type)
        if ven_key: 
            if len(ven_key) == 1:
                # ven = getattr(e,ven_key)
                try: 
                    ven = e.fields_dict[ven_key[0]].value
                except KeyError:
                    ven = ""
            else: 
                
                ven = [e.fields_dict.get(vki,"") for vki in ven_key]
                # ven = [e.value for e in ven if e is not None else ""]
                ven =list(map(lambda x: x.value if x else "", ven))
                
                ven = sorted(ven, key=len)[0]
            yield ven
        continue


set([v for v in venues(library)])

# [e for e in library.entries]

# venues = 
#     [library.entries_dict[i] for i in newdat.index]



